# Sequence Feature Ablation Experiments

Ce notebook reprend le script `sequence_feature_ablation_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste quelles features causales de pose/geometrie sont utiles au modele live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Sequence feature-group ablation for danger prediction.
- Commande de reproduction referencee : feature ablation.
- Artefacts controles : Sequence feature-group ablation exists. (`runs/exp_023_sequence_feature_ablation/metrics/feature_ablation_summary.csv`).
- Run par defaut : `runs/exp_023_sequence_feature_ablation`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_feature_ablation_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch

from ml_pipeline import ROOT, HORIZONS, RUNS_DIR, write_json
from sequence_experiments import append_report, evaluate_catalogue_model, make_run_dir, train_one_model


## Fonction `load_sequence`

Cette cellule definit `load_sequence`. Elle prepare une partie du script.

In [ ]:
def load_sequence(source_run):
    source = Path(source_run)
    if not source.is_absolute():
        source = ROOT / source
    data = np.load(source / "features" / "sequence_dataset.npz")
    X = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    meta = pd.read_csv(source / "features" / "sequence_index.csv")
    config = json.loads((source / "features" / "sequence_feature_columns.json").read_text(encoding="utf-8"))
    return source, X, y, meta, config["feature_columns"]


## Fonction `columns_matching`

Cette cellule definit `columns_matching`. Elle prepare une partie du script.

In [ ]:
def columns_matching(columns, predicate):
    return [idx for idx, col in enumerate(columns) if predicate(col)]


## Fonction `build_feature_groups`

Cette cellule definit `build_feature_groups`. Elle prepare une partie du script.

In [ ]:
def build_feature_groups(columns):
    groups = {
        "all_features": list(range(len(columns))),
        "zone_geometry_only": columns_matching(
            columns,
            lambda c: (
                c in {"max_signed_dist_norm", "any_part_inside"}
                or c.endswith("_signed_dist_norm")
                or c.endswith("_inside")
            ),
        ),
        "zone_motion_only": columns_matching(
            columns,
            lambda c: c in {"max_signed_dist_vel", "max_signed_dist_acc"} or c.endswith("_signed_dist_vel"),
        ),
        "xy_pose_only": columns_matching(
            columns,
            lambda c: c in {"bbox_area_norm", "person_conf"} or c.endswith("_x_norm") or c.endswith("_y_norm") or c.endswith("_conf"),
        ),
        "xy_motion_only": columns_matching(columns, lambda c: c.endswith("_x_vel") or c.endswith("_y_vel")),
        "no_motion": columns_matching(columns, lambda c: not c.endswith("_vel") and not c.endswith("_acc")),
        "no_zone_geometry": columns_matching(
            columns,
            lambda c: (
                c not in {"max_signed_dist_norm", "max_signed_dist_vel", "max_signed_dist_acc", "any_part_inside"}
                and not c.endswith("_signed_dist_norm")
                and not c.endswith("_signed_dist_vel")
                and not c.endswith("_inside")
            ),
        ),
        "hands_arms_geometry_motion": columns_matching(
            columns,
            lambda c: (
                c.startswith("left_wrist_")
                or c.startswith("right_wrist_")
                or c.startswith("left_elbow_")
                or c.startswith("right_elbow_")
                or c in {"max_signed_dist_norm", "max_signed_dist_vel", "max_signed_dist_acc", "any_part_inside"}
            ),
        ),
        "head_torso_geometry_motion": columns_matching(
            columns,
            lambda c: (
                c.startswith("head_")
                or c.startswith("torso_")
                or c in {"max_signed_dist_norm", "max_signed_dist_vel", "max_signed_dist_acc", "any_part_inside"}
            ),
        ),
    }
    return {name: idxs for name, idxs in groups.items() if idxs}


## Fonction `selection_score`

Cette cellule definit `selection_score`. Elle prepare une partie du script.

In [ ]:
def selection_score(row):
    return (
        float(row.get("average_precision", 0) or 0)
        + 0.5 * float(row.get("best_hit_rate", 0) or 0)
        + 0.2 * float(row.get("best_window_precision", 0) or 0)
        - 0.03 * min(float(row.get("best_false_alarms_per_min", 20) or 20), 20.0)
    )


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics, run_dir):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    h1["selection_score"] = h1.apply(selection_score, axis=1)
    h1.to_csv(run_dir / "metrics" / "feature_ablation_horizon1_metrics.csv", index=False)

    rows = []
    for (feature_set, spec, split), group in h1.groupby(["feature_set", "spec", "split"]):
        row = group.iloc[0].to_dict()
        rows.append(
            {
                "feature_set": feature_set,
                "spec": spec,
                "split": split,
                "feature_count": int(row["feature_count"]),
                "average_precision": float(row["average_precision"]),
                "roc_auc": row["roc_auc"],
                "best_threshold": row.get("best_threshold_by_hit_fa"),
                "hit_rate": row.get("best_hit_rate"),
                "false_alarms_per_min": row.get("best_false_alarms_per_min"),
                "window_precision": row.get("best_window_precision"),
                "window_recall": row.get("best_window_recall"),
                "selection_score": float(row["selection_score"]),
            }
        )
    summary = pd.DataFrame(rows)
    summary.to_csv(run_dir / "metrics" / "feature_ablation_summary.csv", index=False)

    val = summary[summary["split"] == "val"].sort_values("selection_score", ascending=False)
    test = summary[summary["split"] == "test"].sort_values("average_precision", ascending=False)

    lines = ["# Sequence Feature-Group Ablation", ""]
    lines.append("Fixed parent-video split from the source sequence run. Every model is a TCN; feature groups are changed to test representation hypotheses.")
    lines.append("")
    lines.append("## Validation Ranking")
    lines.append("")
    lines.append("| rank | feature set | spec | features | AP | hit | FA/min | precision | score |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(val.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['feature_set']} | {row['spec']} | {int(row['feature_count'])} | {row['average_precision']:.3f} | {float(row['hit_rate']):.3f} | {float(row['false_alarms_per_min']):.3f} | {float(row['window_precision']):.3f} | {row['selection_score']:.3f} |"
        )
    lines.append("")
    lines.append("## Test Diagnostics")
    lines.append("")
    lines.append("These rows are diagnostics only; model selection should use validation.")
    lines.append("")
    lines.append("| rank | feature set | spec | features | AP | hit | FA/min | precision |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(test.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['feature_set']} | {row['spec']} | {int(row['feature_count'])} | {row['average_precision']:.3f} | {float(row['hit_rate']):.3f} | {float(row['false_alarms_per_min']):.3f} | {float(row['window_precision']):.3f} |"
        )
    lines.append("")
    best_val = val.iloc[0]
    lines.append("## Interpretation")
    lines.append("")
    lines.append(f"- Best validation configuration: `{best_val['feature_set']}` with `{best_val['spec']}`.")
    lines.append("- Use this as feature-representation evidence, not as a replacement for the 20-seed architecture stability result.")
    lines.append("- If `no_zone_geometry` performs well, the model may be using actor/staging/time cues; if zone geometry groups perform well, the model is using the intended distance-to-danger signal.")
    (run_dir / "feature_ablation_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source, X, y, meta, feature_columns = load_sequence(args.sequence_run)
    run_dir = make_run_dir(args.run_name)
    groups = build_feature_groups(feature_columns)
    if args.groups:
        groups = {name: groups[name] for name in args.groups if name in groups}

    specs = [
        ("tcn_aug_bce", "tcn", True, "bce"),
        ("tcn_aug_focal", "tcn", True, "focal"),
    ]
    if args.include_noaug:
        specs.append(("tcn_noaug_bce", "tcn", False, "bce"))

    config = {
        "source_run": str(source),
        "feature_groups": {name: [feature_columns[i] for i in idxs] for name, idxs in groups.items()},
        "specs": [spec[0] for spec in specs],
        "epochs": args.epochs,
        "patience": args.patience,
        "batch_size": args.batch_size,
        "note": "Fixed parent-video split feature-group ablation. Source sequence data is already train-normalized.",
    }
    write_json(run_dir / "config.json", config)
    write_json(run_dir / "features" / "feature_group_manifest.json", config["feature_groups"])

    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    for group_name, idxs in groups.items():
        X_group = X[:, :, idxs].astype(np.float32)
        for spec_name, kind, augment, loss in specs:
            model_name = f"{group_name}_{spec_name}"
            print(f"training {model_name} ({len(idxs)} features)")
            model_args = SimpleNamespace(
                seed=args.seed,
                batch_size=args.batch_size,
                lr=args.lr,
                weight_decay=args.weight_decay,
                epochs=args.epochs,
                patience=args.patience,
                loss=loss,
                label_smoothing=args.label_smoothing,
                focal_gamma=args.focal_gamma,
            )
            model, history, train_time_s, model_size_bytes = train_one_model(
                model_name, kind, augment, X_group, y, meta, run_dir, model_args, device
            )
            for row in history:
                row["feature_set"] = group_name
                row["feature_count"] = len(idxs)
                row["spec"] = spec_name
            all_history.extend(history)
            rows, sweeps = evaluate_catalogue_model(
                model_name,
                model,
                X_group,
                y,
                meta,
                run_dir,
                device,
                train_time_s,
                model_size_bytes,
                args.batch_size,
                loss,
            )
            for row in rows:
                row["feature_set"] = group_name
                row["feature_count"] = len(idxs)
                row["spec"] = spec_name
            all_metrics.extend(rows)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "feature_ablation_metrics.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "feature_ablation_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "feature_ablation_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "feature_ablation_training_history.csv", index=False)
    summarize(metrics, run_dir)
    append_report(
        run_dir,
        "Feature Ablation Completion",
        f"- Source run: `{source}`\n- Feature groups: `{list(groups)}`\n- Specs: `{[spec[0] for spec in specs]}`\n- Summary: `{run_dir / 'feature_ablation_summary.md'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Sequence feature-group ablation for danger prediction.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_023_sequence_feature_ablation")
    parser.add_argument("--groups", nargs="*", default=[])
    parser.add_argument("--include-noaug", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--epochs", type=int, default=18)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_023_sequence_feature_ablation_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_feature_ablation_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
